In [ ]:
# Import libraries

import warnings
warnings.filterwarnings("ignore")

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import cv2
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as albu
import segmentation_models_pytorch as smp
import segmentation_models_pytorch.utils.metrics
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as BaseDataset
import gc
from pathlib import Path

torch.backends.cudnn.benchmark = True


In [ ]:
# Configure V2 paths and training settings

DATA_ROOT = Path("data") / "V2"
OUTPUT_DIR = Path("results") / "deeplabv3plus" / "V2"

ENCODER = "se_resnext101_32x4d"
ENCODER_WEIGHTS = "imagenet"
CLASSES = ["building"]
BATCH_SIZE = 2
LR = 0.0001
start_epoch = 0
new_epoch = 50
total_epoch = start_epoch + new_epoch
TRAINING = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXPECTED_COUNTS = {"Train": 967, "Validation": 276, "Test": 138}

for subset in EXPECTED_COUNTS:
    image_dir = DATA_ROOT / subset / "Image"
    mask_dir = DATA_ROOT / subset / "Mask"
    if not image_dir.is_dir() or not mask_dir.is_dir():
        raise FileNotFoundError(f"Dataset directories not found: {image_dir}, {mask_dir}")
    image_names = set(os.listdir(image_dir))
    mask_names = set(os.listdir(mask_dir))
    if image_names != mask_names:
        raise ValueError(f"Image and mask filenames do not match in {subset}.")
    if len(image_names) != EXPECTED_COUNTS[subset]:
        raise ValueError(f"{subset}: expected {EXPECTED_COUNTS[subset]} patches, found {len(image_names)}.")
    print(f"V2/{subset}: {len(image_names)} patches")

x_train_dir = os.fspath(DATA_ROOT / "Train" / "Image")
y_train_dir = os.fspath(DATA_ROOT / "Train" / "Mask")
x_valid_dir = os.fspath(DATA_ROOT / "Validation" / "Image")
y_valid_dir = os.fspath(DATA_ROOT / "Validation" / "Mask")
x_test_dir = os.fspath(DATA_ROOT / "Test" / "Image")
y_test_dir = os.fspath(DATA_ROOT / "Test" / "Mask")

model_name = "DeepLabV3Plus"
model_path = os.fspath(OUTPUT_DIR)
predict_path = os.fspath(OUTPUT_DIR / "prediction")
mask_path = os.fspath(OUTPUT_DIR / "prediction" / "masks")
conf_path = os.fspath(OUTPUT_DIR / "confusion")
weight_path = os.fspath(OUTPUT_DIR / "weights")
for directory in (model_path, predict_path, mask_path, conf_path, weight_path):
    Path(directory).mkdir(parents=True, exist_ok=True)

best_model_path = os.fspath(Path(weight_path) / "best_model.pth")
final_model_path = os.fspath(Path(weight_path) / "final_model.pth")
print("Device:", DEVICE)
print("Dataset:", DATA_ROOT)
print("Output:", OUTPUT_DIR)


In [ ]:
# Define visualization helpers

def visualize(save_path=None, **images):
    """Plot images in one row."""
    n = len(images)
    plt.figure(figsize=(16, 5))
    for i, (name, image) in enumerate(images.items()):
        plt.subplot(1, n, i + 1)
        plt.xticks([])
        plt.yticks([])
        plt.title(' '.join(name.split('_')).title())
        plt.imshow(image)
    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()
    plt.close()


def denormalize(x):
    """Scale image to range 0..1 for correct plot"""
    x_max = np.percentile(x, 98)
    x_min = np.percentile(x, 2)
    x = (x - x_min) / (x_max - x_min)
    x = x.clip(0, 1)
    return x


In [ ]:
# Define the reference dataset

class Dataset(BaseDataset):

    CLASSES = ['building']

    def __init__(
            self,
            images_dir,
            masks_dir,
            classes=None,
            augmentation=None,
            preprocessing=None,
    ):
        self.ids = os.listdir(images_dir)
        self.images_fps = [os.path.join(images_dir, image_id) for image_id in self.ids]
        self.masks_fps = [os.path.join(masks_dir, image_id) for image_id in self.ids]


        self.class_values = [self.CLASSES.index(cls.lower()) for cls in classes]

        self.augmentation = augmentation
        self.preprocessing = preprocessing

    def __getitem__(self, i):


        image = cv2.imread(self.images_fps[i])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks_fps[i], 0)


        masks = [(mask == v) for v in self.class_values]
        mask = np.stack(masks, axis=-1).astype('float')


        if self.augmentation:
            sample = self.augmentation(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']


        if self.preprocessing:
            sample = self.preprocessing(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        return image, mask

    def __len__(self):
        return len(self.ids)


In [ ]:
# Define ImageNet preprocessing

def to_tensor(x, **kwargs):
    return x.transpose(2, 0, 1).astype('float32')

def get_preprocessing(preprocessing_fn):

    _transform = [
        albu.Lambda(image=preprocessing_fn),
        albu.Lambda(image=to_tensor, mask=to_tensor),
    ]
    return albu.Compose(_transform)


In [ ]:
# Preserve the reference confusion colors

def conf(gt, predict):
    image_conf = np.zeros([np.shape(gt)[0], np.shape(gt)[1], 3]).astype(np.uint8)
    TP, TN, FP, FN = [0, 255, 0], [0, 0, 0], [255, 0, 0], [0, 0, 255]

    for h in range(np.shape(gt)[0]):
        for w in range(np.shape(gt)[1]):
            mask1_pix = gt[h,w]
            mask2_pix = predict[h,w]

            if mask2_pix==1 and mask1_pix==1:
                image_conf[h,w] = TN

            elif mask2_pix==0 and mask1_pix==0:
                image_conf[h,w] = TP

            elif mask2_pix==1 and mask1_pix==0:
                image_conf[h,w] = FN

            elif mask2_pix==0 and mask1_pix==1:
                image_conf[h,w] = FP
    return image_conf


In [ ]:
# Preview training samples

if __name__ == '__main__':


    dataset = Dataset(x_train_dir, y_train_dir, classes=['building'])
    print("Displaying training samples...")
    for i in range(10):
        image, mask = dataset[i]
        visualize(
            image=image,
            building_mask=mask.squeeze(),
        )
    print("Sample preview completed.")


In [ ]:
# Initialize the ImageNet-pretrained DeepLabV3+

preprocessing_fn = smp.encoders.get_preprocessing_fn(ENCODER, ENCODER_WEIGHTS)
n_classes = 1 if len(CLASSES) == 1 else (len(CLASSES) + 1)
ACTIVATION = 'sigmoid' if n_classes == 1 else 'softmax'
"""
Models: Unet, UnetPlusPlus, FPN, PSPNet, DeepLabV3Plus
"""



model_name = 'DeepLabV3Plus'
model = smp.DeepLabV3Plus(
        encoder_name=ENCODER,
        encoder_weights=ENCODER_WEIGHTS,
        classes=len(CLASSES),
        activation=ACTIVATION,
    )


In [ ]:
# Configure the reference loss, metrics, and optimizer

loss = smp.utils.losses.DiceLoss()


metrics = [smp.utils.metrics.Accuracy(),
               smp.utils.metrics.IoU(threshold=0.5),
               smp.utils.metrics.Fscore(threshold=0.5),
               smp.utils.metrics.Precision(),
               smp.utils.metrics.Recall()]



optimizer = torch.optim.RMSprop([dict(params=model.parameters(), lr=LR),])


In [ ]:
# Create training and validation datasets

train_dataset = Dataset(
        x_train_dir,
        y_train_dir,
        preprocessing=get_preprocessing(preprocessing_fn),
        classes=CLASSES,
    )

valid_dataset = Dataset(
        x_valid_dir,
        y_valid_dir,
        preprocessing=get_preprocessing(preprocessing_fn),
        classes=CLASSES,
    )


In [ ]:
# Create data loaders

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
    drop_last=True
)
valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False
     )


In [ ]:
# Create the reference epoch runners

train_epoch = smp.utils.train.TrainEpoch(
        model,
        loss=loss,
        metrics=metrics,
        optimizer=optimizer,
        device=DEVICE,
        verbose=True,
    )

valid_epoch = smp.utils.train.ValidEpoch(
        model,
        loss=loss,
        metrics=metrics,
        device=DEVICE,
        verbose=True,
    )


In [ ]:
# Define the reference training loop

def train(start_epochs, n_epochs, model, final_model_path, best_model_path, train_dataloader, valid_dataloader, train_epoch, valid_epoch, TRAINING):

    if TRAINING:
        best_iou_score = 0.0
        train_logs_list, valid_logs_list = [], []

        for epoch in range(start_epochs, n_epochs):


            print('\nEpoch: {}'.format(epoch+1))
            train_logs = train_epoch.run(train_dataloader)
            valid_logs = valid_epoch.run(valid_dataloader)



            train_logs_list.append(train_logs)
            valid_logs_list.append(valid_logs)


            if best_iou_score < valid_logs['iou_score']:
                best_iou_score = valid_logs['iou_score']
                torch.save(model, best_model_path)
                print('Best Model saved!')

            if epoch == n_epochs-1:
                torch.save(model, final_model_path)
                print('Final Model saved!')
    return model, train_logs_list, valid_logs_list


In [ ]:
# Train for 50 epochs and save logs

gc.collect()
trained_model,train_logs_list,valid_logs_list = train(start_epoch, total_epoch, model, final_model_path, best_model_path, train_dataloader, valid_dataloader, train_epoch, valid_epoch, TRAINING)


import pandas as pd

train_logs_df = pd.DataFrame(train_logs_list)
valid_logs_df = pd.DataFrame(valid_logs_list)


train_logs_df.columns = ["loss", "accuracy", "iou_score", "fscore", "precision", "recall"]
valid_logs_df.columns = ["val_loss", "val_accuracy", "val_iou_score", "val_fscore", "val_precision", "val_recall"]

logs_df = pd.concat([train_logs_df, valid_logs_df], axis=1)

logs_df.to_csv(model_path + "/train_logs.csv", index=False)


In [ ]:
# Save training and validation metric graphs

graph_groups = (
    ("metric_graph_loss_iou.png", (("loss", "Loss"), ("iou_score", "IoU"))),
    ("metric_graph_f1_acc.png", (("fscore", "F1-score"), ("accuracy", "Accuracy"))),
    ("metric_graph_pre_recall.png", (("precision", "Precision"), ("recall", "Recall"))),
)
for filename, panels in graph_groups:
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    for axis, (metric, title) in zip(axes, panels):
        axis.plot(train_logs_df.index, train_logs_df[metric], label="Train")
        axis.plot(valid_logs_df.index, valid_logs_df[f"val_{metric}"], label="Validation")
        axis.set_title(f"Model {title}")
        axis.set_xlabel("Epoch")
        axis.set_ylabel(title)
        axis.legend(loc="upper left")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename)
    plt.show()
    plt.close(fig)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for axis, metric in zip(axes.flat, ("loss", "accuracy", "iou_score", "fscore", "precision", "recall")):
    axis.plot(train_logs_df.index, train_logs_df[metric], label="Train")
    axis.plot(valid_logs_df.index, valid_logs_df[f"val_{metric}"], label="Validation")
    axis.set_title(metric.replace("_", " ").title())
    axis.set_xlabel("Epoch")
    axis.legend(loc="upper left")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "metric_graph.png")
plt.show()
plt.close(fig)
print("Metric graphs saved:", OUTPUT_DIR)


In [ ]:
# Create the test dataset

test_dataset = Dataset(
        x_test_dir,
        y_test_dir,
        preprocessing=get_preprocessing(preprocessing_fn),
        classes=CLASSES,
    )


In [ ]:
# Evaluate the final checkpoint with the reference metrics

test_dataloader = DataLoader(test_dataset, num_workers=0, pin_memory=True)



model = torch.load(final_model_path, map_location=DEVICE, weights_only=False)


test_epoch = smp.utils.train.ValidEpoch(
        model=model,
        loss=loss,
        metrics=metrics,
        device=DEVICE,
    )

valid_logs = test_epoch.run(test_dataloader)
metric_name = ['accuracy', 'iou_score', 'fscore', 'precision', 'recall']
print(f"Loss: {valid_logs['dice_loss']:.4f}")
print(f"mean accuracy: {valid_logs['accuracy']:.4f}")
print(f"mean iou_score: {valid_logs['iou_score']:.4f}")
print(f"mean fscore: {valid_logs['fscore']:.4f}")
print(f"mean precision: {valid_logs['precision']:.4f}")
print(f"mean recall: {valid_logs['recall']:.4f}")

test_summary = pd.DataFrame([valid_logs])
test_summary.to_csv(OUTPUT_DIR / "test_summary.csv", index=False)
print("Test checkpoint: final epoch", total_epoch)
print("Test patches:", len(test_dataset))
print("Test summary:", OUTPUT_DIR / "test_summary.csv")


In [ ]:
# Save prediction panels, masks, and confusion images

test_dataset_vis = Dataset(
        x_test_dir, y_test_dir,
        classes=['building'],
    )
test_list = os.listdir(x_test_dir)

for i in range(len(test_list)):
        image_vis = test_dataset_vis[i][0].astype('uint8')
        image, gt_mask = test_dataset[i]

        gt_mask = gt_mask.squeeze()

        x_tensor = torch.from_numpy(image).to(DEVICE).unsqueeze(0)
        pr_mask = model.predict(x_tensor)
        pr_mask = (pr_mask.squeeze().cpu().numpy().round())

        image_conf = conf(gt_mask, pr_mask)

        visualize(
            save_path=Path(predict_path) / f"{Path(test_list[i]).stem}.png",
            image=image_vis,
            ground_truth_mask=gt_mask,
            predicted_mask=pr_mask,
            image_conf=image_conf
        )

        cv2.imwrite(mask_path + '/'+ test_list[i], pr_mask)
        cv2.imwrite(conf_path + '/'+ test_list[i], image_conf)
